In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp

In [0]:
df = spark.table("databricks_projects.silver.patient_records_clean")
display(df.take(10))

patient_id,first_name,last_name,full_name,date_of_birth,age,gender,email,address,city,state,admission_date,discharge_date,length_of_stay_days,admission_type,department,attending_physician,primary_diagnosis,secondary_diagnosis,medication_1,medication_2,height_cm,weight_kg,bmi,heart_rate_bpm,temperature_celsius,oxygen_saturation_pct,blood_glucose_mgdl,creatinine_mgdl,hba1c_pct,insurance_provider,insurance_claim_amount_usd,icu_days,readmission_30day_flag,smoker,alcohol_use,_silver_timestamp
10089,Christopher,Brown,Christopher Brown,1951-09-15,70,Male,christopher.brown91@aol.com,3658 Hill Way,Fort Worth,KY,2021-10-27,2021-11-19,23,Transfer,Oncology,Dr. Hill,COPD,Alzheimer's Disease,Sertraline,null,null,57.0,17.3,97.0,38.6,97.0,null,3.02,null,Centene,33590.32,5,0,Yes,null,2026-05-06T00:53:14.403086Z
10150,Donna,Carter,Donna Carter,1977-01-27,41,Female,donna.carter60@gmail.com,8166 Pine Ln,Philadelphia,OR,2018-08-11,2018-08-18,7,Transfer,Oncology,Dr. Jones,Epilepsy,null,Aspirin,Albuterol,184.2,62.3,18.36,112.0,39.9,97.0,192.0,5.82,6.8,United Health,32913.02,3,1,No,Heavy,2026-05-06T00:53:14.403086Z
10366,Dorothy,null,Dorothy,1974-05-14,50,Male,dorothy.robinson63@yahoo.com,5328 Park Ln,Indianapolis,CA,2024-09-15,2024-09-22,7,Urgent,Psychiatry,Dr. Carter,Atrial Fibrillation,Depression,Sertraline,null,191.6,null,23.8,100.0,38.7,95.0,null,1.96,4.7,null,50668.93,4,0,null,Heavy,2026-05-06T00:53:14.403086Z
10648,null,Campbell,Campbell,1980-10-12,42,Male,carol.campbell65@yahoo.com,1104 Main Ln,Baltimore,AZ,2023-08-20,2023-08-29,9,Routine,Surgery,Dr. Adams,Asthma,Pneumonia,Amoxicillin,Albuterol,null,108.8,43.8,89.0,null,98.0,null,5.4,10.1,Bluecross Blueshield,10009.51,3,1,null,Moderate,2026-05-06T00:53:14.403086Z
10650,Kimberly,Martinez,Kimberly Martinez,1988-11-03,32,Female,kimberly.martinez42@yahoo.com,5745 Oak Ln,Philadelphia,OK,2020-11-10,2020-12-09,29,Routine,ICU,Dr. Hall,Chronic Kidney Disease,Anemia,Amlodipine,null,187.9,null,24.9,111.0,38.6,100.0,314.0,3.64,6.3,Medicare,26408.67,4,1,No,None,2026-05-06T00:53:14.403086Z
10696,Kimberly,Martinez,Kimberly Martinez,1995-02-11,26,Female,kimberly.martinez72@gmail.com,6422 Maple St,Milwaukee,NJ,2021-03-28,null,null,Urgent,ICU,Dr. Lee,Sepsis,Coronary Artery Disease,Warfarin,null,166.7,123.5,44.44,117.0,36.3,null,118.0,2.0,12.7,Bluecross Blueshield,25623.49,0,0,No,Occasional,2026-05-06T00:53:14.403086Z
10735,John,Adams,John Adams,1962-10-07,60,Female,john.adams4@aol.com,2899 Maple St,Las Vegas,SC,2023-08-22,null,null,Elective,Psychiatry,Dr. Mitchell,Hypothyroidism,Chronic Kidney Disease,Lisinopril,null,167.9,129.6,45.97,53.0,39.7,97.0,null,4.18,8.3,Kaiser Permanente,8736.68,0,0,Yes,Moderate,2026-05-06T00:53:14.403086Z
10771,Ashley,Jackson,Ashley Jackson,1964-03-03,56,Male,ashley.jackson77@gmail.com,3933 Hill Dr,El Paso,MI,2020-10-22,2020-10-27,5,Transfer,ICU,Dr. Wilson,Parkinson's Disease,COPD,Albuterol,Atorvastatin,184.1,52.8,15.58,113.0,36.9,90.0,329.0,5.53,5.7,null,12696.67,0,0,No,None,2026-05-06T00:53:14.403086Z
10879,Joseph,Campbell,Joseph Campbell,1992-11-17,31,Female,joseph.campbell51@outlook.com,8646 Washington Ave,Portland,OH,2024-05-14,2024-05-30,16,Elective,Orthopedics,Dr. Johnson,Osteoarthritis,Appendicitis,Insulin Glargine,Amlodipine,185.8,61.6,17.84,94.0,39.8,null,200.0,5.9,6.4,Kaiser Permanente,80612.31,0,0,Yes,null,2026-05-06T00:53:14.403086Z
11100,Karen,Roberts,Karen Roberts,1954-04-18,68,Male,null,6768 Maple Rd,Boston,IN,2023-04-05,null,null,Transfer,Orthopedics,Dr. Green,Asthma,Atrial Fibrillation,Atorvastatin,Amlodipine,163.7,104.1,38.85,79.0,39.4,89.0,129.0,4.68,10.2,Anthem,64898.17,0,0,No,Heavy,2026-05-06T00:53:14.403086Z


Create table

In [0]:
%skip
%sql

CREATE or REPLACE TABLE gold.patient_records_final
USING DELTA
AS SELECT * FROM silver.patient_records_clean LIMIT 0;

num_affected_rows,num_inserted_rows


Merge Logic

In [0]:
from delta.tables import DeltaTable

target_table = "gold.patient_records_final"

delta_table = DeltaTable.forName(spark, target_table)

(delta_table.alias("tgt")
 .merge(
     df.alias("src"),
     "tgt.patient_id = src.patient_id"
 )
 .whenMatchedUpdate(
      condition="tgt._silver_timestamp < src._silver_timestamp",
      set={
     "first_name": "src.first_name",
     "last_name": "src.last_name",
     "full_name": "src.full_name",
     "date_of_birth": "src.date_of_birth",
     "age": "src.age",
     "gender": "src.gender",
     "email": "src.email",
     "address": "src.address",
     "city": "src.city",
     "state": "src.state",
     "admission_date": "src.admission_date",
     "discharge_date": "src.discharge_date",
     "length_of_stay_days": "src.length_of_stay_days",
     "department": "src.department",
     "primary_diagnosis": "src.primary_diagnosis",
     "_silver_timestamp": F.current_timestamp()
 })
 .whenNotMatchedInsertAll()
 .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:

print(dbutils.secrets.list('PRecord'))

SQL_SERVER   = dbutils.secrets.get(scope='PRecord', key="ServerName")
SQL_DATABASE = dbutils.secrets.get(scope='PRecord', key="DBName")
SQL_USER     = dbutils.secrets.get(scope='PRecord', key="UserName")
SQL_PASSWORD = dbutils.secrets.get(scope='PRecord', key="AzSQLPass")

JDBC_URL = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;"
    f"database={SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

JDBC_PROPS = {
    "user":             SQL_USER,
    "password":         SQL_PASSWORD,
    "driver":           "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "batchsize":        "10000",
    "queryTimeout":     "0",
    "reliabilityLevel": "BEST_EFFORT",
    "tableLock":        "true",
}

[SecretMetadata(key='AzSQLPass'), SecretMetadata(key='DBName'), SecretMetadata(key='ServerName'), SecretMetadata(key='UserName')]


**Load into SQL DB (Serving Layer)**

In [0]:
gold_df = spark.table("gold.patient_records_final")

gold_df.write \
  .format("jdbc") \
  .option("url", JDBC_URL) \
  .option("dbtable", "dbo.patient_records_stg") \
  .options(**JDBC_PROPS) \
  .mode("overwrite") \
  .save()

In [0]:
%skip
(
    df.write
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "dbo.patient_records_clean")
    .options(**JDBC_PROPS)
    .mode("overwrite")
    .save()
)